# Study 899 — Cash + Call "90/10" — the teardown

The three-book race (90/10 / buy-and-hold / matched static), the excess-of-cash identity, the leverage-clean **convexity** spanning alpha, the block-bootstrap Sharpe-difference CI, the two-era cut, the **premium (variance-risk-premium) sweep** that is the tradability crux, and the synthetic control. The 10% sleeve is a Black–Scholes-marked 1-yr ATM SPY call — a documented proxy for a listed call (realized-vol priced, no dividend: both tilts *flatter* the strategy, named on the Signal axis).

In [1]:
R = {'alpha_ann': 0.46,
 'asym': -0.003,
 'avg_w': 0.68,
 'bear_prot': 0.369,
 'bear_prot_min': 0.043,
 'bear_w': 0.41,
 'beta': 0.638,
 'bh_cagr': 10.7,
 'bh_dd': -55.2,
 'bh_sharpe': 0.543,
 'bh_sortino': 0.513,
 'bh_vol': 19.78,
 'boot_hi': 0.21,
 'boot_lo': -0.334,
 'boot_point': -0.051,
 'boot_win': 33.8,
 'calm_w': 0.79,
 'crash08_bh': -36.8,
 'crash08_bh_dd': -47.1,
 'crash08_c': -8.1,
 'crash08_c_dd': -10.2,
 'crash20_bh': 18.3,
 'crash20_bh_dd': -33.7,
 'crash20_c': -1.1,
 'crash20_c_dd': -15.7,
 'crash22_bh': -18.2,
 'crash22_bh_dd': -24.5,
 'crash22_c': -17.1,
 'crash22_c_dd': -19.6,
 'diff_t_nw': -1.37,
 'dn_cap': 0.568,
 'end': '2026-06-30',
 'era_e_bh': 0.335,
 'era_e_n': 2163,
 'era_e_t': 0.94,
 'era_e_tt': 0.453,
 'era_e_vs': 0.119,
 'era_l_bh': 0.767,
 'era_l_n': 2635,
 'era_l_t': 1.68,
 'era_l_tt': 0.999,
 'era_l_vs': 0.232,
 'fp': 'b954e845292f',
 'n_days': 4799,
 'n_rolls': 19,
 'n_seeds': 30,
 'pm080_sh': 0.629,
 'pm080_vs': 0.087,
 'pm100_sh': 0.492,
 'pm100_vs': -0.051,
 'pm125_sh': 0.335,
 'pm125_vs': -0.208,
 'pm150_sh': 0.193,
 'pm150_vs': -0.35,
 'pm200_sh': -0.039,
 'pm200_vs': -0.582,
 'sharpe_vs_bh': -0.051,
 'start': '2007-05-31',
 'static_cagr': 8.09,
 'static_dd': -40.2,
 'static_sharpe': 0.543,
 'static_vol': 13.44,
 't_alpha': 0.33,
 'tt_cagr': 6.13,
 'tt_dd': -19.8,
 'tt_sharpe': 0.492,
 'tt_sortino': 0.497,
 'tt_vol': 10.42,
 'turnover': 0.18,
 'up_cap': 0.564}

## The race — 90/10 vs buy-and-hold vs matched static mix (excess-of-cash, gross, BS-fair)

A *constant* fraction of SPY funded from cash has the **same** excess-of-cash Sharpe as SPY — so the matched-static and buy-and-hold Sharpes coincide, and 90/10's only distinguishing act is the option's **convexity** (the spanning alpha).

In [2]:
print(f"90/10       : exSharpe {R['tt_sharpe']:.3f}  Sortino {R['tt_sortino']:.3f}  CAGR {R['tt_cagr']:5.2f}%  vol {R['tt_vol']:5.2f}%  maxDD {R['tt_dd']:.1f}%")
print(f"buy-and-hold: exSharpe {R['bh_sharpe']:.3f}  Sortino {R['bh_sortino']:.3f}  CAGR {R['bh_cagr']:5.2f}%  vol {R['bh_vol']:5.2f}%  maxDD {R['bh_dd']:.1f}%")
print(f"static@avgw : exSharpe {R['static_sharpe']:.3f}                CAGR {R['static_cagr']:5.2f}%  vol {R['static_vol']:5.2f}%  maxDD {R['static_dd']:.1f}%")
print(f"vs buy-and-hold {R['sharpe_vs_bh']:+.3f}   convexity alpha {R['alpha_ann']:+.2f}%/yr (HAC t {R['t_alpha']:+.2f}, beta {R['beta']:.3f})")
print(f"up-capture {R['up_cap']:.3f} / down-capture {R['dn_cap']:.3f} (asym {R['asym']:+.3f})  avg Δ-w {R['avg_w']:.2f}  roll turnover {R['turnover']:.2f}x/yr")

90/10       : exSharpe 0.492  Sortino 0.497  CAGR  6.13%  vol 10.42%  maxDD -19.8%
buy-and-hold: exSharpe 0.543  Sortino 0.513  CAGR 10.70%  vol 19.78%  maxDD -55.2%
static@avgw : exSharpe 0.543                CAGR  8.09%  vol 13.44%  maxDD -40.2%
vs buy-and-hold -0.051   convexity alpha +0.46%/yr (HAC t +0.33, beta 0.638)
up-capture 0.564 / down-capture 0.568 (asym -0.003)  avg Δ-w 0.68  roll turnover 0.18x/yr


## Bootstrap — circular block CI on the excess-Sharpe difference (90/10 − buy-and-hold)

In [3]:
print(f"gain {R['boot_point']:+.3f}  95% CI [{R['boot_lo']:+.3f}, {R['boot_hi']:+.3f}]  "
      f"P(90/10 wins) {R['boot_win']:.1f}%   -> the CI straddles zero: a statistical tie")

gain -0.051  95% CI [-0.334, +0.210]  P(90/10 wins) 33.8%   -> the CI straddles zero: a statistical tie


## Crash years — capital protection bites, the recovery is rented not owned

In [4]:
for yr,bh,bd,c,cd in [(2008,R['crash08_bh'],R['crash08_bh_dd'],R['crash08_c'],R['crash08_c_dd']),
                      (2020,R['crash20_bh'],R['crash20_bh_dd'],R['crash20_c'],R['crash20_c_dd']),
                      (2022,R['crash22_bh'],R['crash22_bh_dd'],R['crash22_c'],R['crash22_c_dd'])]:
    print(f"{yr}: BH {bh:+6.1f}% (DD {bd:6.1f}%)  ->  90/10 {c:+6.1f}% (DD {cd:6.1f}%)")

2008: BH  -36.8% (DD  -47.1%)  ->  90/10   -8.1% (DD  -10.2%)
2020: BH  +18.3% (DD  -33.7%)  ->  90/10   -1.1% (DD  -15.7%)
2022: BH  -18.2% (DD  -24.5%)  ->  90/10  -17.1% (DD  -19.6%)


## Robustness — two eras (split 2016-01-01)

Each half is *individually* a shade favourable to 90/10 (vs-BH +0.12, +0.23) yet the pooled gap is slightly negative — the usual Sharpe-not-additive artefact of mixing a high-vol GFC regime with a calmer one. Neither era's convexity alpha clears significance (*t* < 2).

In [5]:
print(f"2007-2015 (n={R['era_e_n']}): 90/10-Sh {R['era_e_tt']:+.3f}  BH-Sh {R['era_e_bh']:+.3f}  vs-BH {R['era_e_vs']:+.3f}  alpha-t {R['era_e_t']:+.2f}")
print(f"2016-2026 (n={R['era_l_n']}): 90/10-Sh {R['era_l_tt']:+.3f}  BH-Sh {R['era_l_bh']:+.3f}  vs-BH {R['era_l_vs']:+.3f}  alpha-t {R['era_l_t']:+.2f}")

2007-2015 (n=2163): 90/10-Sh +0.453  BH-Sh +0.335  vs-BH +0.119  alpha-t +0.94
2016-2026 (n=2635): 90/10-Sh +0.999  BH-Sh +0.767  vs-BH +0.232  alpha-t +1.68


## The tradability crux — the premium (variance risk premium) sweep

The Black–Scholes price uses **realized** vol; a real listed call trades at **implied** vol (IV/RV ≈ 1.1–1.4 on the S&P — the variance risk premium). `prem_mult` scales the option cost to that reality; the near-tie evaporates the moment you pay what the option costs.

In [6]:
for lbl,sh,vs in [('0.80x (too cheap)',R['pm080_sh'],R['pm080_vs']),('1.00x BS-fair',R['pm100_sh'],R['pm100_vs']),
                  ('1.25x mild VRP',R['pm125_sh'],R['pm125_vs']),('1.50x typical',R['pm150_sh'],R['pm150_vs']),
                  ('2.00x stressed',R['pm200_sh'],R['pm200_vs'])]:
    print(f"  {lbl:18s}: 90/10 Sharpe {sh:+.3f}   vs S&P {vs:+.3f}")

  0.80x (too cheap) : 90/10 Sharpe +0.629   vs S&P +0.087
  1.00x BS-fair     : 90/10 Sharpe +0.492   vs S&P -0.051
  1.25x mild VRP    : 90/10 Sharpe +0.335   vs S&P -0.208
  1.50x typical     : 90/10 Sharpe +0.193   vs S&P -0.350
  2.00x stressed    : 90/10 Sharpe -0.039   vs S&P -0.582


## Synthetic control — the machinery is unbiased (live, offline)

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cash_call import data, strategy as st
bear = np.array([st.synthetic_detect(data.synthetic_prices(seed=899+s, n_days=2500, drift=-0.0004, sigma=0.016)[0])['dd_protection'] for s in range(8)])
bw   = np.array([st.synthetic_detect(data.synthetic_prices(seed=899+s, n_days=2500, drift=-0.0004, sigma=0.016)[0])['avg_weight'] for s in range(8)])
cw   = np.array([st.synthetic_detect(data.synthetic_prices(seed=899+s, n_days=2500, drift=0.0005, sigma=0.006)[0])['avg_weight'] for s in range(8)])
print(f"bear (8 seeds): drawdown protection mean {bear.mean():+.3f}  (capital floored)")
print(f"equity weight: bear {bw.mean():.2f}  <  calm {cw.mean():.2f}  -> the rule de-risks as vol rises")

bear (8 seeds): drawdown protection mean +0.419  (capital floored)
equity weight: bear 0.38  <  calm 0.77  -> the rule de-risks as vol rises


## Verdict

- **Signal — WEAK.** Capital protection is **real and mechanical**: maxDD **-19.8%** vs **-55.2%** at half the vol, 2008 cut −47%→−10%, confirmed by a 30-seed synthetic control. But the risk-adjusted-return claim fails: even at the BS-fair premium the excess Sharpe **0.492 vs 0.543** is a tie (bootstrap CI [-0.334, +0.210]), the convexity adds no alpha (*t* = +0.33), and both eras are insignificant. Single GFC-anchored ~19-yr window; the BS mark *flatters* (realized-vol, no dividend).
- **Tradability — MIRAGE.** The parity is an artefact of the fair-price assumption. A real call carries the variance risk premium (IV>RV): at a **1.25–1.5×** markup the excess Sharpe drops to **-0.208…-0.350**, clearly below buy-and-hold — and the call-holder forgoes the ~1.8%/yr dividend. Not a friction story (0.18×/yr turnover). At best it *matches* buy-and-hold's Sharpe while giving up ~4.6%/yr CAGR — a payoff reshaping, not a paycheck.